# 02 — EDA và baseline chất lượng
Thống kê từng cột và phân bố các trường cần thiết. Null thật và chuỗi rỗng được đo riêng. Các phép groupBy có thể shuffle và chạy lâu trên dữ liệu thật.

In [ ]:
from pathlib import Path
import os
import sys

candidate = Path(os.environ.get("SEC_PROJECT_ROOT", Path.cwd())).resolve()
locations = (Path.cwd().resolve(), Path.cwd().resolve().parent, candidate)
code_root = next((p for p in locations if (p / "src").is_dir()), None)
if code_root is None:
    raise RuntimeError("Hãy mở notebook từ sec_spark_project hoặc thư mục notebooks của dự án")
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
os.environ.setdefault("SEC_PROJECT_ROOT", str(code_root))
from src.config import make_spark, DATA, REPORTS, QUARTERS, TABLES
spark = make_spark()
print("Spark", spark.version, "|", spark.sparkContext.master, "|", QUARTERS)


In [ ]:
from src.quality import eda
summaries = eda(spark)
summaries

## Xem bảng trước xử lý
Chỉ đưa bảng thống kê nhỏ về pandas. Không dùng `toPandas` trên dữ liệu NUM.

In [ ]:
import json
import pandas as pd
nulls = pd.DataFrame(json.loads((REPORTS / "profile_bronze_columns.json").read_text(encoding="utf-8")))
nulls.sort_values(["table", "missing_including_blank_pct"], ascending=[True, False])

In [ ]:
for column in ("form", "fy", "prevrpt", "uom"):
    print("Phân bố", column)
    spark.read.option("header", True).csv(str(REPORTS / "eda" / column)).show(20, truncate=False)

## Câu hỏi diễn giải
Vì sao coreg/footnote có tỷ lệ trống cao? Vì sao số trùng theo khóa đề bài lớn hơn khóa SEC? Không suy ra tất cả các giá trị thiếu là lỗi. Xem chính sách trong `docs/cleaning_decisions.md`.

In [ ]:
spark.stop()